In [2]:
# Install deps if needed (run once)
import subprocess, sys
try:
    import mlx_tune, datasets, dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx-tune", "datasets", "python-dotenv", "openai", "groq", "requests", "beautifulsoup4", "lxml", "tqdm"])

In [3]:
# Load API keys from .env file
import os
from dotenv import load_dotenv
load_dotenv()

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
MOONSHOT_API_KEY = os.getenv('MOONSHOT_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

print('DEEPSEEK_API_KEY set:', bool(DEEPSEEK_API_KEY))
print('MOONSHOT_API_KEY set:', bool(MOONSHOT_API_KEY))
print('GROQ_API_KEY set:', bool(GROQ_API_KEY))
print('HF_TOKEN set:', bool(HF_TOKEN))


KIMI_API_KEY: Not set
GROQ_API_KEY: Set
KIMI_BASE_URL: Not set


## Phase 2: Generate Training Data (AutoDidact)

Generates 10k+ Q&A pairs grounded in real regulatory documents (PMLA 2002, FIU-IND enforcement orders, SEBI reports).
Uses Groq llama-3.3-70b as teacher LLM — reads source chunks, generates natural exam-style Q&A.


## Phase 2b: AutoDidact Generation (FIU-IND + SEBI grounded)

Generates 10k Q&A pairs grounded in real regulatory documents (PMLA 2002, FIU-IND enforcement orders, SEBI reports) using Groq llama-3.3-70b as teacher LLM.


In [ ]:
import sys
sys.path.insert(0, 'fintech_data')
from autodidact_gen import AutoDidactGenerator
from autodidact_fetcher import AutoDidactFetcher
import json

# Fetch source docs (cached after first run)
fetcher = AutoDidactFetcher()
source = fetcher.fetch_all()
all_docs = source['fiu_ind'] + source['sebi']
print(f'Source docs: {len(all_docs)}')

# Generate 10k grounded Q&A pairs
gen = AutoDidactGenerator(
    model='llama-3.3-70b-versatile',
    cache_file='data/fintech_data_autodidact.json',
)
pairs = gen.generate_from_docs(all_docs, target_pairs=10000, retries=5)
print(f'Generated {len(pairs)} autodidact pairs')


## Phase 4: Process & Merge Data

Combine all data sources, filter quality, split into train/eval, format for Qwen chat template.


In [ ]:
from fintech_data import DataProcessor
import os, json

if 'processor' not in globals():
    processor = DataProcessor()

# Load autodidact grounded QA pairs (FIU-IND + SEBI)
autodidact_path = os.path.join("data", "fintech_data_autodidact.json")
autodidact_pairs = []
if os.path.exists(autodidact_path):
    with open(autodidact_path, encoding="utf-8") as f:
        autodidact_pairs = json.load(f)
    print(f"Loaded autodidact: {len(autodidact_pairs)} pairs")

# Load the reliable base dataset (HF finance QA + RBI master direction PDFs)
reliable_path = os.path.join("data", "fintech_data_reliable.json")
reliable_pairs = []
if os.path.exists(reliable_path):
    with open(reliable_path, encoding="utf-8") as f:
        reliable_pairs = json.load(f)
    print(f"Loaded reliable base: {len(reliable_pairs)} pairs")

# Merge all sources
all_pairs = autodidact_pairs + reliable_pairs
all_pairs = processor.filter_quality(all_pairs)
print(f"Total pairs after merge: {len(all_pairs)}")

formatted = processor.format_for_training(all_pairs, model_type="qwen")
train_data, eval_data = processor.split_dataset(formatted, train_ratio=0.85)
processor.save_split(train_data, eval_data)


## Phase 5: LoRA Fine-Tuning

Applies LoRA to Qwen2.5-1.5B. Only ~0.5% of parameters are trained (adapter weights).

In [ ]:
from fintech_data import FintechFineTuner, DataProcessor

if 'processor' not in globals():
    processor = DataProcessor()

train_data, eval_data = processor.load_dataset()
print(f"Loaded: {len(train_data)} train, {len(eval_data)} eval")

if 'model_name' not in globals():
    model_name = "mlx-community/Qwen2.5-1.5B-4bit"

tuner = FintechFineTuner(
    base_model_name=model_name,
    output_dir="fintech_finetuned_qwen",
    max_seq_length=512,
)

tuner.load_base_model()
tuner.apply_lora(r=16, lora_alpha=32)

tuner.train(
    train_data=train_data,
    eval_data=eval_data,
    num_epochs=3,
    batch_size=2,
    learning_rate=2e-4,
    max_seq_length=512,
)


## Phase 6: Test the Fine-Tuned Model

In [ ]:
test_questions = [
    "What is CKYC and how is it different from regular KYC?",
    "What are the steps for CERSAI registration?",
    "What is the difference between VKYC and e-KYC?",
    "What is AML and what are the reporting obligations for a fintech?",
    "How does the PMLA 2002 affect financial institutions?",
]

for q in test_questions:
    print(f"
{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    response = tuner.inference(q)
    print(f"A: {response}")
